# Lab 22.2: Did the Explanation Help? Analyzing a Reliance Study

Chapter 22 of *Explainable AI in Practice*. You analyze a **simulated** user study: 180 participants, randomized to three conditions, each deciding 24 two-option cases with help from an AI that is right on 75 percent of cases.

- **A: AI only.** The AI's answer, no explanation.
- **B: Attribution.** The AI's answer with a feature-attribution chart. In this simulation it raises the tendency to follow the AI, whether or not the AI is right (the pattern reported by Bansal et al., 2021).
- **C: Verifiable.** The AI's answer with an explanation that lets the participant check it. Checking succeeds more often when the AI is wrong (the mechanism argued by Vasconcelos et al., 2023), at a cost in effort.

Every participant also gave a satisfaction rating from 1 to 7. Nothing here is a real person; the data are generated with a fixed seed so your numbers match the checkpoints.

Run-only path: run every cell and read the outputs. Coding path: complete the two exercises at the end.

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(22)
N_PER, N_CASES, AI_ACC, OWN_ACC = 60, 24, 0.75, 0.68

def simulate(condition):
    rows = []
    for pid in range(N_PER):
        follow = {"A": 0.50, "B": 0.72, "C": 0.45}[condition] + rng.normal(0, 0.08)
        for case in range(N_CASES):
            ai_right = rng.random() < AI_ACC
            own_right = rng.random() < OWN_ACC
            if condition == "C" and rng.random() < (0.55 if not ai_right else 0.15):
                # the participant checks the explanation and sees whether the AI is right
                human, agree = True, ai_right
            elif rng.random() < follow:
                human, agree = ai_right, True
            else:
                human, agree = own_right, own_right == ai_right
            rows.append(dict(condition=condition, pid=f"{condition}{pid:02d}", case=case,
                             ai_right=ai_right, agree=agree, correct=bool(human)))
    return rows

df = pd.DataFrame(simulate("A") + simulate("B") + simulate("C"))
sat_mean = {"A": 4.1, "B": 5.6, "C": 4.4}
sat = pd.DataFrame({"pid": df.pid.unique()})
sat["condition"] = sat.pid.str[0]
sat["satisfaction"] = np.clip(np.round([rng.normal(sat_mean[c], 1.0) for c in sat.condition]), 1, 7)
print(df.shape, sat.shape)
df.head()

(4320, 6) (180, 3)


,condition,pid,case,ai_right,agree,correct
0,A,A00,0,True,True,True
1,A,A00,1,True,False,False
2,A,A00,2,False,False,True
3,A,A00,3,True,True,True
4,A,A00,4,True,True,True


## Step 1: Measures per participant

For each participant compute: accuracy (share of correct decisions), **overreliance** (share of wrong-AI cases where they agreed with the AI) and **underreliance** (share of right-AI cases where they disagreed).

In [2]:
def per_participant(g):
    return pd.Series({
        "accuracy": g.correct.mean(),
        "overreliance": g.agree[~g.ai_right].mean(),
        "underreliance": 1 - g.agree[g.ai_right].mean(),
        "agreement": g.agree.mean(),
    })

pp = df.groupby(["condition", "pid"]).apply(per_participant, include_groups=False).reset_index()
pp = pp.merge(sat[["pid", "satisfaction"]], on="pid")
pp.groupby("condition")[["accuracy", "overreliance", "underreliance", "agreement", "satisfaction"]].mean().round(3)

,accuracy,overreliance,underreliance,agreement,satisfaction
condition,,,,,
A,0.722,0.667,0.162,0.792,4.100
B,0.736,0.784,0.098,0.872,5.633
C,0.820,0.309,0.145,0.719,4.483


**Checkpoint 1.** Condition B has the highest agreement, the highest overreliance and the highest satisfaction. Condition C has the lowest overreliance and the highest accuracy, and its satisfaction is close to A's.

## Step 2: Bootstrap intervals

Resample participants (not cases) with replacement, because cases from the same person are not independent.

In [3]:
def boot_ci(x, B=4000, seed=0):
    r = np.random.default_rng(seed)
    x = np.asarray(x)
    m = r.choice(x, (B, len(x))).mean(axis=1)
    return np.quantile(m, [0.025, 0.975])

rows = []
for c, g in pp.groupby("condition"):
    for m in ["accuracy", "overreliance", "underreliance", "satisfaction"]:
        lo, hi = boot_ci(g[m])
        rows.append(dict(condition=c, measure=m, mean=g[m].mean(), low=lo, high=hi))
ci = pd.DataFrame(rows).round(3)
ci.pivot(index="measure", columns="condition", values=["mean", "low", "high"])

mean                  low                 high              
condition          A      B      C      A      B      C      A      B      C
measure                                                                     
accuracy       0.722  0.736  0.820  0.702  0.714  0.799  0.742  0.758  0.841
overreliance   0.667  0.784  0.309  0.612  0.734  0.262  0.724  0.833  0.355
satisfaction   4.100  5.633  4.483  3.850  5.417  4.183  4.367  5.867  4.783
underreliance  0.162  0.098  0.145  0.140  0.078  0.124  0.184  0.120  0.168

## Step 3: Difference between conditions

The decision question is whether C beats A and B on accuracy and overreliance. Bootstrap the difference of means.

In [4]:
def diff_ci(a, b, B=4000, seed=1):
    r = np.random.default_rng(seed)
    a, b = np.asarray(a), np.asarray(b)
    d = r.choice(a, (B, len(a))).mean(axis=1) - r.choice(b, (B, len(b))).mean(axis=1)
    return a.mean() - b.mean(), *np.quantile(d, [0.025, 0.975])

for m in ["accuracy", "overreliance"]:
    for x, y in [("C", "A"), ("C", "B"), ("B", "A")]:
        est, lo, hi = diff_ci(pp[pp.condition == x][m], pp[pp.condition == y][m])
        print(f"{m:13s} {x} - {y}: {est:+.3f}  95% interval [{lo:+.3f}, {hi:+.3f}]")

accuracy      C - A: +0.098  95% interval [+0.070, +0.126]
accuracy      C - B: +0.084  95% interval [+0.053, +0.115]
accuracy      B - A: +0.014  95% interval [-0.016, +0.043]
overreliance  C - A: -0.359  95% interval [-0.432, -0.285]
overreliance  C - B: -0.476  95% interval [-0.540, -0.408]
overreliance  B - A: +0.117  95% interval [+0.041, +0.191]


**Checkpoint 2.** B minus A on overreliance is clearly positive (the interval excludes zero); C minus B on overreliance is clearly negative; C minus A on accuracy is positive.

## Step 4: Does satisfaction predict good reliance?

Correlate each participant's satisfaction with their overreliance, within and across conditions.

In [5]:
print("all participants:", round(pp.satisfaction.corr(pp.overreliance), 3))
for c, g in pp.groupby("condition"):
    print(c, round(g.satisfaction.corr(g.overreliance), 3))
pp.groupby("condition")[["satisfaction", "overreliance"]].mean().round(2)

all participants: 0.236
A -0.069
B 0.264
C 0.091


,satisfaction,overreliance
condition,,
A,4.10,0.67
B,5.63,0.78
C,4.48,0.31


Across conditions, the condition people liked most (B) is the one with the most overreliance. Satisfaction is not evidence of usefulness (Chapter 2, Chapter 22).

## Exercises (coding path)

1. Add a **calibration** measure: for each participant, agreement rate on right-AI cases minus agreement rate on wrong-AI cases. Which condition has the best calibration?
2. Rerun the simulation with `AI_ACC = 0.95`. How many wrong-AI cases does each participant now see, and how wide are the overreliance intervals? What does this say about designing a study with a very accurate model?

## Write-up

One page: the three measures with intervals per condition, satisfaction, and your recommendation on which explanation to ship, with one sentence on what a real study would need that this simulation lacks.

In [6]:
# Exercise 1 (one possible solution, shown so that run-only readers see the result)
calib = df.groupby(["condition", "pid"]).apply(
    lambda g: g.agree[g.ai_right].mean() - g.agree[~g.ai_right].mean(), include_groups=False)
calib.groupby("condition").mean().round(3)

condition
A    0.171
B    0.117
C    0.546
dtype: float64